In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# from pathlib import Path

# SRC = Path('/content/drive/MyDrive')

# # the manifest: tries your filename first, then the original one
# manifest_candidates = ['paperstyle_fdf_epoch.csv', 'fdf_epoch_manifest.csv', 'paperstyle_fdf_epoch_manifest.csv']
# MANIFEST_SRC = next((SRC / n for n in manifest_candidates if (SRC / n).exists()), None)

# required = ['deep_utils.py', 'epochs.mat', 'channel_names.csv',
#             'bad_channel_audit.csv', 'ica_log.csv', 'metadata.csv']

# print('Manifest:', MANIFEST_SRC)
# missing = [f for f in required if not (SRC / f).exists()]
# if MANIFEST_SRC is None:
#     missing.append('manifest csv')
# print('Missing:', missing if missing else 'nothing, good to go')

In [3]:
# import sys, shutil, zipfile
# import pandas as pd

# ROOT = Path('/content/c3_picard_50HzNotch_FDF_epochs')
# D = ROOT / 'C3_picard_50HzNotch_FDF' / 'win8_0'
# D.mkdir(parents=True, exist_ok=True)

# # get epochs.mat onto Colab's local disk (faster than reading from Drive)
# if not (D / 'epochs.mat').exists():
#     if (SRC / 'epochs.mat').exists():
#         shutil.copy(SRC / 'epochs.mat', D / 'epochs.mat')
#     elif (SRC / 'epochs.zip').exists():
#         with zipfile.ZipFile(SRC / 'epochs.zip') as z:
#             z.extractall(D)
#     else:
#         raise FileNotFoundError('Neither epochs.mat nor epochs.zip found in MyDrive')

# for f in ['metadata.csv', 'bad_channel_audit.csv', 'ica_log.csv', 'channel_names.csv']:
#     shutil.copy(SRC / f, D / f)

# shutil.copy(SRC / 'deep_utils.py', '/content/deep_utils.py')
# if '/content' not in sys.path:
#     sys.path.insert(0, '/content')

# mf = pd.read_csv(MANIFEST_SRC)
# for col, name in [('epoch_mat_path', 'epochs.mat'), ('metadata_path', 'metadata.csv'),
#                   ('audit_path', 'bad_channel_audit.csv'), ('ica_log_path', 'ica_log.csv'),
#                   ('channel_names_path', 'channel_names.csv')]:
#     mf[col] = str(D / name)
# mf.to_csv(ROOT / 'paperstyle_fdf_epoch_manifest.csv', index=False)

# print(sorted(p.name for p in D.iterdir()))
# print('metadata rows:', len(pd.read_csv(D / 'metadata.csv')), '(should be 390000)')
import sys, shutil
import pandas as pd
from pathlib import Path

INP = Path('/kaggle/input')
def find(name):
    hits = list(INP.rglob(name))
    return hits[0] if hits else None

ROOT = Path('/kaggle/working/epochs_root')
D    = ROOT / 'C3_picard_50HzNotch_FDF' / 'win8_0'
D.mkdir(parents=True, exist_ok=True)

MANIFEST_SRC = next((p for n in ['paperstyle_fdf_epoch.csv', 'fdf_epoch_manifest.csv']
                     if (p := find(n)) is not None), None)
assert MANIFEST_SRC is not None, 'manifest csv not found'

for f in ['epochs.mat', 'metadata.csv', 'bad_channel_audit.csv', 'ica_log.csv', 'channel_names.csv']:
    src = find(f)
    assert src is not None, f'{f} not found in /kaggle/input'
    if not (D / f).exists():
        shutil.copy(src, D / f)

shutil.copy(find('deep_utils.py'), '/kaggle/working/deep_utils.py')
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

mf = pd.read_csv(MANIFEST_SRC)
for col, name in [('epoch_mat_path', 'epochs.mat'), ('metadata_path', 'metadata.csv'),
                  ('audit_path', 'bad_channel_audit.csv'), ('ica_log_path', 'ica_log.csv'),
                  ('channel_names_path', 'channel_names.csv')]:
    mf[col] = str(D / name)
mf.to_csv(ROOT / 'paperstyle_fdf_epoch_manifest.csv', index=False)
print('ready:', sorted(p.name for p in D.iterdir()))

ready: ['bad_channel_audit.csv', 'channel_names.csv', 'epochs.mat', 'ica_log.csv', 'metadata.csv']


In [4]:
# print(sorted(p.name for p in ROOT.iterdir()))
# print(pd.read_csv(ROOT / 'paperstyle_fdf_epoch_manifest.csv')[['variant','mode','n_samples','n_channels','n_time']])
m = pd.read_csv(ROOT / 'paperstyle_fdf_epoch_manifest.csv')
print(m[['preprocessing', 'feature_mode', 'n_samples', 'n_channels', 'n_time']])
print(m['epoch_mat_path'].iloc[0])

             preprocessing feature_mode  n_samples  n_channels  n_time
0  C3_picard_50HzNotch_FDF       win8_0     390000          15       8
/kaggle/working/epochs_root/C3_picard_50HzNotch_FDF/win8_0/epochs.mat


In [5]:
from pathlib import Path
from IPython.display import clear_output, display
import time
import warnings
import sys
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")


for helper_dir in [
    # Path('/content'),
    Path('/kaggle/working'),
    Path.cwd(),
    Path.cwd() / 'python',
]:
    if helper_dir.exists() and str(helper_dir) not in sys.path:
        sys.path.insert(0, str(helper_dir))

from deep_utils import (
    set_global_seed, get_device, ensure_dir,
    load_manifest, load_epoch_dataset,
    make_fold_indices, audit_split,
    fit_adaptive_zscore, apply_adaptive_zscore,
    build_model as _utils_build_model, train_model, evaluate_trial_level,
    append_row_csv, read_results, result_exists,
    summarize_results, save_best_by_calibration, cleanup_device,
    RESULT_COLUMNS, AUDIT_COLUMNS
)

class Config:
    epoch_root = Path('/kaggle/working/epochs_root')

    desktop_dir = Path('/kaggle/working')
    output_dir = Path('/kaggle/working/results')

    seed = 42
    prefer_xpu = True

    max_epochs = 500
    patience = 30
    batch_size = 256
    base_lr = 1e-3
    weight_decay = 1e-4

    calib_lr = 1e-3
    calib_patience = 30
    calib_epochs_1trial = 180
    calib_epochs_5trials = 260

    pred_batch_size = 512

cfg = Config()
cfg.desktop_dir.mkdir(parents=True, exist_ok=True)
ensure_dir(cfg.output_dir)

set_global_seed(cfg.seed)
device = get_device(cfg.prefer_xpu)

print('Device:', device)
print('Epoch root:', cfg.epoch_root)
print('Output dir:', cfg.output_dir)

RESULT_FILE = cfg.output_dir / 'all_fold_results.csv'
HISTORY_FILE = cfg.output_dir / 'training_history.csv'
AUDIT_FILE = cfg.output_dir / 'split_leakage_audit.csv'
PRED_FILE = cfg.output_dir / 'fold_trial_predictions.csv'
SUMMARY_FILE = cfg.output_dir / 'summary.csv'
BEST_FILE = cfg.output_dir / 'best_by_calibration.csv'

FIG_ACCURACY_COMPARISON = cfg.output_dir / 'fig_01_model_accuracy_comparison.png'
FIG_SIGNIFICANCE_MATRIX = cfg.output_dir / 'fig_02_model_significance_matrix.png'
FIG_BEST_MODEL_DECLARATION = cfg.output_dir / 'fig_03_best_model_declaration.png'

FRIEDMAN_FILE = cfg.output_dir / 'friedman_summary.csv'
NEMENYI_FILE = cfg.output_dir / 'nemenyi_posthoc.csv'
NADEAU_BENGIO_FILE = cfg.output_dir / 'nadeau_bengio_pairwise.csv'
MODEL_MEAN_SUMMARY_FILE = cfg.output_dir / 'model_mean_accuracy_summary.csv'
PARAM_COUNT_FILE = cfg.output_dir / 'model_parameter_counts.csv'

Device: cuda
Epoch root: /kaggle/working/epochs_root
Output dir: /kaggle/working/results


In [6]:
class CNNLSTMFamily(nn.Module):
    """
    CNN/LSTM family for EEG windows.

    Input shape: B x C x T
    C = EEG channels, T = time samples per window.
    """
    def __init__(
        self,
        n_channels: int,
        n_times: int,
        n_classes: int,
        n_lstm_layers: int = 1,
        bidirectional: bool = False,
        use_cnn: bool = True,
        conv_filters: int = 64,
        hidden: int = 64,
        dropout: float = 0.35,
    ):
        super().__init__()
        self.use_cnn = use_cnn
        self.bidirectional = bidirectional
        self.n_lstm_layers = n_lstm_layers

        if use_cnn:
            k = min(7, max(3, n_times if n_times % 2 == 1 else n_times - 1))
            self.conv = nn.Sequential(
                nn.Conv1d(n_channels, conv_filters, kernel_size=k, padding=k // 2, bias=False),
                nn.BatchNorm1d(conv_filters),
                nn.ELU(),
                nn.MaxPool1d(kernel_size=2) if n_times >= 4 else nn.Identity(),
                nn.Dropout(dropout),
            )
            lstm_input = conv_filters
        else:
            self.conv = None
            lstm_input = n_channels

        self.lstm = nn.LSTM(
            input_size=lstm_input,
            hidden_size=hidden,
            num_layers=n_lstm_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_lstm_layers > 1 else 0.0,
        )

        out_dim = hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, n_classes)
        )

    def forward(self, x):
        if self.use_cnn:
            z = self.conv(x)
            z = z.transpose(1, 2)
        else:
            z = x.transpose(1, 2)

        out, _ = self.lstm(z)
        last = out[:, -1, :]
        return self.head(last)


MODEL_NAMES = [
    'CNN-1-LSTM',
    'CNN-2-LSTM',
    'CNN-2-Bi-LSTM',
    'CNN-3-LSTM',
    '3-LSTM',
]

MODEL_COLORS = {
    'CNN-1-LSTM': '#4C78A8',
    'CNN-2-LSTM': '#F58518',
    'CNN-2-Bi-LSTM': '#54A24B',
    'CNN-3-LSTM': '#B279A2',
    '3-LSTM': '#E45756',
}


def build_model(name: str, n_channels: int, n_times: int, n_classes: int) -> nn.Module:
    """
    Local model factory. This overrides the imported utility build_model to ensure all five
    required model names are available in this notebook.
    """
    name = str(name)

    if name == 'CNN-1-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=1, bidirectional=False, use_cnn=True)

    if name == 'CNN-2-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=2, bidirectional=False, use_cnn=True)

    if name == 'CNN-2-Bi-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=2, bidirectional=True, use_cnn=True)

    if name == 'CNN-3-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=3, bidirectional=False, use_cnn=True)

    if name == '3-LSTM':
        return CNNLSTMFamily(n_channels, n_times, n_classes, n_lstm_layers=3, bidirectional=False, use_cnn=False)

    return _utils_build_model(name, n_channels, n_times, n_classes)


In [7]:
import deep_utils
_orig_load_epoch_dataset = deep_utils.load_epoch_dataset

def load_epoch_dataset(row):
    ds = _orig_load_epoch_dataset(row)
    n_ch = int(row['n_channels'])            # 15, from the manifest
    n_t  = int(row['n_time'])                # 8
    if ds.X.shape[1] == n_t and ds.X.shape[2] == n_ch and n_ch != n_t:
        ds.X = np.ascontiguousarray(ds.X.transpose(0, 2, 1))   # N,T,C -> N,C,T
        ds.window_sec = ds.X.shape[2] / ds.fs
    assert ds.X.shape[1:] == (n_ch, n_t), f'unexpected shape {ds.X.shape}'
    return ds

In [8]:
MODEL_NAMES = ['CNN-1-LSTM']

In [ ]:
manifest = load_manifest(cfg.epoch_root)
manifest = manifest[manifest["mode"].astype(str).str.contains("win8")]
manifest = manifest[manifest["variant"].astype(str).str.contains("FDF")]
manifest = manifest[~manifest["variant"].astype(str).str.contains("noFDF")]
if len(manifest) == 0:
    raise RuntimeError(
        "No FD-F win8_0 rows found in the manifest after filtering. Check that "
        "epoch_root points at the right folder and that the FD-F export ran."
    )
display(manifest)

CALIBRATIONS = [
    # ('pure_loso', 0),
    # ('calib_approx5pct_1trial', 1),
    # ('calib_20pct_5trials', 5),
    #('calib_40pct_10trials', 10),
    ('calib_12pct_3trials', 3),
    ('calib_28pct_7trials', 7),
]
CALIB_DISPLAY_NAMES = {
    'pure_loso': 'Pure LOSO',
    'calib_approx5pct_1trial': '1-trial calibration',
    'calib_20pct_5trials': '5-trial calibration',
    'calib_12pct_3trials': '3-trial calibration',
    'calib_28pct_7trials': '7-trial calibration',
    'calib_40pct_10trials': '10-trial calibration',
}
CALIB_ORDER = [c[0] for c in CALIBRATIONS]

plan = []
for _, row in manifest.iterrows():
    for model_name in MODEL_NAMES:
        for calib_name, trials_per_class in CALIBRATIONS:
            plan.append(dict(
                row=row.to_dict(),
                variant=str(row['variant']),
                mode=str(row['mode']),
                model=model_name,
                calibration_name=calib_name,
                calib_trials_per_class=trials_per_class,
            ))

_first_ds_for_count = load_epoch_dataset(pd.Series(plan[0]['row']))
N_SUBJECTS = len(sorted(np.unique(_first_ds_for_count.participants).tolist()))
EXPECTED_FOLDS = len(plan) * N_SUBJECTS
N_CHANNELS_GLOBAL = _first_ds_for_count.X.shape[1]
N_TIMES_GLOBAL = _first_ds_for_count.X.shape[2]
N_CLASSES_GLOBAL = len(_first_ds_for_count.class_names)
print('Models:', MODEL_NAMES)
print('Calibration settings:', CALIB_ORDER)
print('Number of subjects detected:', N_SUBJECTS)
print('Plan items before subjects:', len(plan))
print('Expected total LOSO folds:', EXPECTED_FOLDS)
print('n_channels:', N_CHANNELS_GLOBAL, ' n_times:', N_TIMES_GLOBAL, ' n_classes:', N_CLASSES_GLOBAL)
del _first_ds_for_count


In [10]:
def read_df(path, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=columns)
    try:
        return pd.read_csv(path)
    except Exception as e:
        print('Could not read', path, e)
        return pd.DataFrame(columns=columns)

def show_progress(current=''):
    done = read_df(RESULT_FILE, RESULT_COLUMNS)
    n_done = len(done)
    total = EXPECTED_FOLDS if 'EXPECTED_FOLDS' in globals() else max(1, n_done)
    pct = 100.0 * n_done / max(1, total)
    bar_len = 40
    filled = int(round(bar_len * pct / 100))
    bar = '█' * filled + '░' * (bar_len - filled)
    print(f'Overall progress: {n_done}/{total} folds = {pct:.2f}%')
    print(bar)
    if current:
        print('Current:', current)
    if len(done):
        tmp = done.groupby(['variant','mode','model','calibration_name'], dropna=False).agg(
            n_folds=('subject','nunique'),
            mean_balanced_accuracy=('balanced_accuracy','mean'),
            mean_accuracy=('accuracy','mean'),
            mean_macro_f1=('macro_f1','mean'),
        ).reset_index().sort_values('mean_balanced_accuracy', ascending=False)
        display(tmp.head(25))


In [11]:

for plan_i, item in enumerate(plan, 1):
    row = pd.Series(item['row'])
    mode = item['mode']
    variant = item['variant']
    model_name = item['model']
    calib_name = item['calibration_name']
    calib_trials_per_class = int(item['calib_trials_per_class'])

    ds = load_epoch_dataset(row)
    subjects = sorted(np.unique(ds.participants).tolist())
    n_classes = len(ds.class_names)
    n_channels = ds.X.shape[1]
    n_times = ds.X.shape[2]
    window_sec = ds.window_sec

    for heldout in subjects:
        if result_exists(RESULT_FILE, mode, model_name, calib_name, heldout):
            continue

        clear_output(wait=True)
        show_progress(
            f'[{plan_i}/{len(plan)}] {variant} | {mode} | {model_name} | '
            f'{calib_name} | heldout={heldout} | train separately'
        )
        start_total = time.time()

        split = make_fold_indices(ds, heldout, calib_trials_per_class, seed=cfg.seed)
        audit = audit_split(ds, heldout, split, calib_name, model_name)
        append_row_csv(AUDIT_FILE, audit, AUDIT_COLUMNS)

        train_idx = split['train_idx']
        val_idx = split['val_idx']
        calib_idx = split['calib_idx']
        test_idx = split['test_idx']

        mean, std = fit_adaptive_zscore(ds.X[train_idx])
        X_train = apply_adaptive_zscore(ds.X[train_idx], mean, std)
        y_train = ds.y[train_idx]
        X_val = apply_adaptive_zscore(ds.X[val_idx], mean, std) if len(val_idx) else None
        y_val = ds.y[val_idx] if len(val_idx) else None
        X_test = apply_adaptive_zscore(ds.X[test_idx], mean, std)
        y_test = ds.y[test_idx]
        tu_test = ds.trial_uid[test_idx]

        set_global_seed(cfg.seed)

        model = build_model(model_name, n_channels, n_times, n_classes)
        model, info = train_model(
            model, X_train, y_train, X_val, y_val, device,
            max_epochs=cfg.max_epochs, patience=cfg.patience, batch_size=cfg.batch_size,
            lr=cfg.base_lr, weight_decay=cfg.weight_decay, verbose=False,
        )
        train_runtime = float(info.get('runtime_sec', 0.0))
        calib_runtime = 0.0

        for h in info.get('history', []):
            append_row_csv(HISTORY_FILE, dict(
                variant=variant, mode=mode, model=model_name, calibration_name=calib_name,
                subject=heldout, phase='source', epoch=h['epoch'], train_loss=h['train_loss'],
                val_accuracy=h.get('val_accuracy', ''), val_balanced_accuracy=h.get('val_balanced_accuracy', ''),
                val_macro_f1=h.get('val_macro_f1', ''),
            ), columns=['variant','mode','model','calibration_name','subject','phase','epoch',
                       'train_loss','val_accuracy','val_balanced_accuracy','val_macro_f1'])

        if len(calib_idx) > 0:
            X_calib = apply_adaptive_zscore(ds.X[calib_idx], mean, std)
            y_calib = ds.y[calib_idx]
            calib_epochs = cfg.calib_epochs_5trials if calib_trials_per_class >= 5 else cfg.calib_epochs_1trial
            model, cinfo = train_model(
                model, X_calib, y_calib, X_val, y_val, device,
                max_epochs=calib_epochs, patience=cfg.calib_patience, batch_size=min(cfg.batch_size, 256),
                lr=cfg.calib_lr, weight_decay=cfg.weight_decay, verbose=False,
            )
            calib_runtime = float(cinfo.get('runtime_sec', 0.0))
            for h in cinfo.get('history', []):
                append_row_csv(HISTORY_FILE, dict(
                    variant=variant, mode=mode, model=model_name, calibration_name=calib_name,
                    subject=heldout, phase='calibration', epoch=h['epoch'], train_loss=h['train_loss'],
                    val_accuracy=h.get('val_accuracy', ''), val_balanced_accuracy=h.get('val_balanced_accuracy', ''),
                    val_macro_f1=h.get('val_macro_f1', ''),
                ), columns=['variant','mode','model','calibration_name','subject','phase','epoch',
                           'train_loss','val_accuracy','val_balanced_accuracy','val_macro_f1'])

        metrics, pred_df = evaluate_trial_level(model, X_test, y_test, tu_test, device, batch_size=cfg.pred_batch_size)
        total_runtime = time.time() - start_total

        result_row = dict(
            pipeline='C3_Picard_50HzNotch_FDF_MultiModel',
            variant=variant,
            mode=mode,
            window_sec=window_sec,
            model=model_name,
            calibration_name=calib_name,
            calib_trials_per_class=calib_trials_per_class,
            subject=heldout,
            n_train_subjects=len(split['train_subjects']),
            n_val_subjects=len(split['val_subjects']),
            n_train_windows=len(train_idx),
            n_val_windows=len(val_idx),
            n_calib_trials=audit['n_calib_trials'],
            n_calib_windows=len(calib_idx),
            n_test_trials=audit['n_test_trials'],
            n_test_windows=len(test_idx),
            accuracy=metrics['accuracy'],
            balanced_accuracy=metrics['balanced_accuracy'],
            macro_f1=metrics['macro_f1'],
            best_epoch=info.get('best_epoch', ''),
            epochs_run=info.get('epochs_run', ''),
            train_runtime_sec=train_runtime,
            calib_runtime_sec=calib_runtime,
            total_runtime_sec=total_runtime,
            device=str(device),
            seed=cfg.seed,
        )
        append_row_csv(RESULT_FILE, result_row, RESULT_COLUMNS)

        if len(pred_df):
            pred_df = pred_df.copy()
            pred_df['pipeline'] = 'C3_Picard_50HzNotch_FDF_MultiModel'
            pred_df['variant'] = variant
            pred_df['mode'] = mode
            pred_df['model'] = model_name
            pred_df['calibration_name'] = calib_name
            pred_df['subject'] = heldout
            pred_df['label_true'] = pred_df['y_true'].map(lambda i: ds.class_names[int(i)] if int(i) < len(ds.class_names) else str(i))
            pred_df['label_pred'] = pred_df['y_pred'].map(lambda i: ds.class_names[int(i)] if int(i) < len(ds.class_names) else str(i))
            header = not Path(PRED_FILE).exists()
            pred_df.to_csv(PRED_FILE, mode='a', header=header, index=False)

        cleanup_device(device)

clear_output(wait=True)
show_progress('C3 Picard + 50 Hz Notch + FD-F multi-model LOSO run completed')
summary = summarize_results(RESULT_FILE, SUMMARY_FILE)
best = save_best_by_calibration(summary, BEST_FILE)
print('Saved:', SUMMARY_FILE)
print('Saved:', BEST_FILE)
display(best)


Overall progress: 0/40 folds = 0.00%
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
Current: [1/2] C3_picard_50HzNotch_FDF | win8_0 | CNN-1-LSTM | calib_12pct_3trials | heldout=P01 | train separately


KeyboardInterrupt: 

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/epochs_root', ignore_errors=True)
print('Cleaned up. Results remain in /kaggle/working/results')